In [ ]:
from google.colab import drive
import os

if not os.path.ismount('/content/drive'):
    # Clear any existing content at the mount point (use with caution!)
    !rm -rf /content/drive
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted!")

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/BrainTumorMRIDataset.zip"  # Update if needed
extract_path = "/content/BrainTumorMRIDataset"

# Extract the ZIP file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extraction complete!")
print("Extracted files:", os.listdir(extract_path))

In [ ]:
import os
import shutil
import random
import zipfile
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

# Set paths
ZIP_FILE_PATH = "/content/drive/MyDrive/BrainTumorMRIDataset.zip"  # Update if needed
EXTRACTED_DATASET_PATH = "/content/BrainTumorMRIDataset"
BALANCED_DATASET_PATH = "/content/drive/MyDrive/BalancedBrainTumorDataset"

# Step 1: Extract dataset if not already extracted
if not os.path.exists(EXTRACTED_DATASET_PATH):
    print("📂 Extracting dataset...")
    with zipfile.ZipFile(ZIP_FILE_PATH, "r") as zip_ref:
        zip_ref.extractall("/content")
    print("✅ Extraction complete!")

# Define class names
classes = ["glioma", "meningioma", "pituitary", "notumor"]

# Set balanced count (smallest class count)
BALANCED_COUNT = 1321  # Adjust based on available images

# Define dataset splits
train_ratio, val_ratio, test_ratio = 0.7, 0.15, 0.15
train_count = int(BALANCED_COUNT * train_ratio)  # 925 images per class
val_count = int(BALANCED_COUNT * val_ratio)      # 198 images per class
test_count = int(BALANCED_COUNT * test_ratio)    # 198 images per class

# Create folders
for split in ["Training", "Validation", "Testing"]:
    for class_name in classes:
        os.makedirs(os.path.join(BALANCED_DATASET_PATH, split, class_name), exist_ok=True)

# Function to copy images (with folder existence check)
def copy_images(src_folder, dest_folder, num_images):
    if not os.path.exists(src_folder):
        print(f"❌ Folder not found: {src_folder}. Skipping...")
        return

    all_images = [f for f in os.listdir(src_folder) if os.path.isfile(os.path.join(src_folder, f))]
    if len(all_images) == 0:
        print(f"❌ No images found in {src_folder}. Skipping...")
        return

    selected_images = random.sample(all_images, min(len(all_images), num_images))
    for img in selected_images:
        shutil.copy(os.path.join(src_folder, img), os.path.join(dest_folder, img))

# Function to augment images
def augment_images(dest_folder, target_count):
    datagen = ImageDataGenerator(
        rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
        shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode="nearest"
    )

    images = [f for f in os.listdir(dest_folder) if os.path.isfile(os.path.join(dest_folder, f))]
    num_existing = len(images)

    if num_existing >= target_count:
        return  # No need to augment

    num_to_generate = target_count - num_existing
    generated_count = 0

    while generated_count < num_to_generate:
        img_name = random.choice(images)
        img = load_img(os.path.join(dest_folder, img_name), target_size=(224, 224))
        img_array = img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        for batch in datagen.flow(img_array, batch_size=1, save_to_dir=dest_folder, save_prefix="aug", save_format="jpeg"):
            generated_count += 1
            if generated_count >= num_to_generate:
                break

# Process each class
for class_name in classes:
    src_train = os.path.join(EXTRACTED_DATASET_PATH, "Training", class_name)
    src_test = os.path.join(EXTRACTED_DATASET_PATH, "Testing", class_name)

    dst_train = os.path.join(BALANCED_DATASET_PATH, "Training", class_name)
    dst_val = os.path.join(BALANCED_DATASET_PATH, "Validation", class_name)
    dst_test = os.path.join(BALANCED_DATASET_PATH, "Testing", class_name)

    # Copy and augment training images
    copy_images(src_train, dst_train, train_count)
    augment_images(dst_train, train_count)

    # Copy validation images
    copy_images(src_train, dst_val, val_count)

    # Copy testing images
    copy_images(src_test, dst_test, test_count)

print("✅ Dataset is perfectly balanced!")

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# --- Configuration ---
DATASET_PATH = "/content/drive/MyDrive/BalancedBrainTumorDataset"
IMG_SIZE = (380, 380)  # Required input size for EfficientNetB4
BATCH_SIZE = 16
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "notumor"]

# --- Data Generators ---
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Training"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Validation"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Testing"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# --- Class Weights ---
y_train = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# --- Load EfficientNetB4 Base ---
base_model = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(380, 380, 3))
base_model.trainable = False  # Phase 1: Freeze base

# --- Build Model ---
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(len(CLASS_NAMES), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# --- Callbacks ---
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

# --- Phase 1: Train Head ---
print("\n🔧 Training top layers...\n")
history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# --- Phase 2: Fine-tune Base ---
print("\n🧠 Fine-tuning EfficientNetB4...\n")
base_model.trainable = True
for layer in base_model.layers[:-20]:  # Unfreeze last 20 layers
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# --- Save the Model ---
model.save("efficientnetb4_brain_tumor.keras")


In [ ]:
# --- Evaluate on Test Set ---
loss, acc = model.evaluate(test_generator)
print(f"\n✅ Final EfficientNetB4 Test Accuracy: {acc:.4f}")


In [ ]:
# --- Plot Training History ---
history = {k: history1.history.get(k, []) + history2.history.get(k, []) for k in set(history1.history) | set(history2.history)}

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['accuracy'], label='Train Acc')
plt.plot(history['val_accuracy'], label='Val Acc')
plt.title("Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title("Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

# --- Classification Report + Confusion Matrix ---
y_true = test_generator.classes
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n📊 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from itertools import cycle

# --- Binarize the true labels ---
n_classes = len(CLASS_NAMES)
y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

# --- Predict Probabilities ---
y_score = model.predict(test_generator)

# --- Compute ROC curve and ROC area for each class ---
fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# --- Plot all ROC curves ---
plt.figure(figsize=(10, 8))
colors = cycle(['blue', 'red', 'green', 'orange'])

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f"{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.2f})")

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-Class ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
import os
import glob

# --- Configuration ---
MODEL_PATH = "efficientnetb4_brain_tumor.keras"
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "notumor"]
IMG_SIZE = (380, 380)

# --- Load the trained model ---
model = load_model(MODEL_PATH)

# --- Function to predict a single image ---
def predict_image(img_path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)
    pred_index = np.argmax(preds)
    pred_class = CLASS_NAMES[pred_index]
    confidence = np.max(preds)

    # Display
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Prediction: {pred_class} ({confidence:.2%})")
    plt.show()

    return pred_class, confidence

# --- Function to predict all images in a folder ---
def predict_folder(folder_path):
    image_paths = glob.glob(os.path.join(folder_path, "*"))
    for img_path in image_paths:
        print(f"\n🖼️ Predicting: {os.path.basename(img_path)}")
        predict_image(img_path)

# --- Example Usage ---

# Predict a single image
# predict_image("/content/drive/MyDrive/BalancedBrainTumorDataset/Testing/glioma/image1.jpg")

# Predict all images from a folder
predict_folder("/content/drive/MyDrive/BalancedBrainTumorDataset/Testing/glioma")


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# --- Configuration ---
DATASET_PATH = "/content/drive/MyDrive/BalancedBrainTumorDataset"
IMG_SIZE = (300, 300)  # EfficientNetB3 default input size
BATCH_SIZE = 16
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "notumor"]

# --- Data Generators ---
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Training"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)
val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Validation"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)
test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Testing"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# --- Class Weights ---
y_train = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# --- Load EfficientNetB3 Base ---
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = False  # Freeze for phase 1

# --- Build Model ---
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(len(CLASS_NAMES), activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

# --- Callbacks ---
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)

# --- Phase 1: Train Head ---
print("\n🔧 Training top layers...\n")
history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_HEAD,
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# --- Phase 2: Fine-tune Base ---
print("\n🧠 Fine-tuning EfficientNetB3...\n")
base_model.trainable = True
for layer in base_model.layers[:-20]:  # Fine-tune last 20 layers
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

history2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_FINE,
    class_weight=class_weights_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# --- Save the Model ---
model.save("efficientnetb3_brain_tumor.keras")


In [ ]:
# --- Evaluate on Test Set ---
loss, acc = model.evaluate(test_generator)
print(f"\n✅ Final EfficientNetB3 Test Accuracy: {acc:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.applications.efficientnet import preprocess_input
import os

# --- Assumptions ---
# model: already trained and saved
# history1, history2: history from training head and fine-tuning
# CLASS_NAMES: defined
# test_generator: already defined (ImageDataGenerator with shuffle=False)

# --- Load Model (optional if not in memory) ---
model = load_model("efficientnetb3_brain_tumor.keras")

# --- Combine History ---
def plot_accuracy_loss(history1, history2):
    acc = history1.history['accuracy'] + history2.history['accuracy']
    val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
    loss = history1.history['loss'] + history2.history['loss']
    val_loss = history1.history['val_loss'] + history2.history['val_loss']

    epochs_range = range(len(acc))
    plt.figure(figsize=(14, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Train Accuracy')
    plt.plot(epochs_range, val_acc, label='Val Accuracy')
    plt.title("Accuracy over Epochs")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Train Loss')
    plt.plot(epochs_range, val_loss, label='Val Loss')
    plt.title("Loss over Epochs")
    plt.legend()

    plt.tight_layout()
    plt.show()

plot_accuracy_loss(history1, history2)

# --- Predictions ---
y_true = test_generator.classes
y_probs = model.predict(test_generator)
y_pred = np.argmax(y_probs, axis=1)

# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# --- Classification Report ---
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# --- ROC Curve (One-vs-Rest for Multi-class) ---
y_test_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
fpr, tpr, roc_auc = {}, {}, {}

plt.figure(figsize=(10, 7))
for i in range(len(CLASS_NAMES)):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    plt.plot(fpr[i], tpr[i], label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Chance')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(True)
plt.show()

# --- Predict Sample Images ---
def predict_image(path, class_names, model, img_size=(300, 300)):
    img = load_img(path, target_size=img_size)
    img_array = img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    class_idx = np.argmax(prediction)
    confidence = np.max(prediction)

    plt.imshow(load_img(path))
    plt.axis('off')
    plt.title(f"Predicted: {class_names[class_idx]} ({confidence:.2%})")
    plt.show()

# --- Example Usage ---
# predict_image("/content/sample.jpg", CLASS_NAMES, model)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input
import os
import glob

# --- Configuration ---
MODEL_PATH = "efficientnetb3_brain_tumor.keras"  # EfficientNetB3 model
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "notumor"]
IMG_SIZE = (300, 300)  # EfficientNetB3 uses 300x300 input size

# --- Load the trained EfficientNetB3 model ---
model = load_model(MODEL_PATH)

# --- Predict a single image ---
def predict_image(img_path):
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)
    pred_index = np.argmax(preds)
    pred_class = CLASS_NAMES[pred_index]
    confidence = np.max(preds)

    # Display the image and prediction
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Prediction: {pred_class} ({confidence:.2%})")
    plt.show()

    return pred_class, confidence

# --- Predict all images in a folder ---
def predict_folder(folder_path):
    image_paths = glob.glob(os.path.join(folder_path, "*"))
    for img_path in image_paths:
        print(f"\n🖼️ Predicting: {os.path.basename(img_path)}")
        predict_image(img_path)

# --- Example Usage ---

# Predict one image
# predict_image("/content/drive/MyDrive/BalancedBrainTumorDataset/Testing/meningioma/image1.jpg")

# Predict all images in a folder (update path to the class folder you want)
predict_folder("/content/drive/MyDrive/BalancedBrainTumorDataset/Testing/glioma")


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# --- Configuration ---
DATASET_PATH = "/content/drive/MyDrive/BalancedBrainTumorDataset"
IMG_SIZE = (380, 380)  # Required input size for EfficientNetB4
BATCH_SIZE = 16
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "notumor"]

# --- Data Generators ---
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Training"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Validation"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Testing"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# --- Class Weights ---
y_train = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# --- Load EfficientNetB4 Base ---
base_model = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(380, 380, 3))
base_model.trainable = False  # Phase 1: Freeze base

# --- Build Model ---
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(len(CLASS_NAMES), activation='softmax')
])
model.summary()

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.regularizers import l2
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# --- Configuration ---
DATASET_PATH = "/content/drive/MyDrive/BalancedBrainTumorDataset"
IMG_SIZE = (300, 300)  # EfficientNetB3 default input size
BATCH_SIZE = 16
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
CLASS_NAMES = ["glioma", "meningioma", "pituitary", "notumor"]

# --- Data Generators ---
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True,
    fill_mode='nearest'
)
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Training"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)
val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Validation"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)
test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, "Testing"),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# --- Class Weights ---
y_train = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

# --- Load EfficientNetB3 Base ---
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = False  # Freeze for phase 1

# --- Build Model ---
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    BatchNormalization(),
    Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(len(CLASS_NAMES), activation='softmax')
])
model.summary()